## 事前準備

In [3]:
import pandas as pd
import datetime
import sqlite3
import pymysql
import pandas.io.sql as psql
from datetime import datetime as dt
import numpy as np
import pandas.tseries.offsets as offsets
import sqlalchemy as sqa
import matplotlib.pyplot as plt
import python_ss as ps
import gspread
from oauth2client.service_account import ServiceAccountCredentials
import json

import os
import ast
import db_dtypes
from google.cloud import bigquery
from google.oauth2 import service_account
from google.cloud import secretmanager

In [4]:
def access_secret_version(project_id, secret_id, version_id='latest'):
    client = secretmanager.SecretManagerServiceClient()

    name = f"projects/{project_id}/secrets/{secret_id}/versions/{version_id}"
    response = client.access_secret_version(request={"name": name})
    payload = response.payload.data.decode("UTF-8")
    return ast.literal_eval(payload)

In [5]:
# 上記関数を実行するコードが記載されています。こちらもそのままお使いください。
credentials = service_account.Credentials.from_service_account_info(
  access_secret_version('temp-for-sandbox', 'TEMP_CREDENTIAL_KEY'),
  scopes=["https://www.googleapis.com/auth/cloud-platform"],)

C:\Users\suehara\Anaconda3\lib\site-packages\google\auth\_default.py:78: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [6]:
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

## SMAP候補者一覧をつくる

In [7]:
##kohoshaテーブルのSMAPだけだとヌケモレ発生する恐れがあるので、
##kohoshaテーブル　or　syokikoshosテーブルの　アポソースが　SMAPとなっている候補者idを重複削除し、それをSMAP候補者一覧とする

In [11]:
query = """
SELECT
  format_date('%Y/%m/%d',created) as created,
  id as kohosha_id,
  seimei,
  consts.name as ap_source
FROM `temp-380708.live_rhs.kohoshas` khs
left join (SELECT 
              code,
              name
             FROM `temp-380708.live_rhs.sys_consts`
             where group_code = 19) consts on khs.ap_source = consts.code
where consts.name = "SMAP"
order by created
"""

client = bigquery.Client(credentials=credentials, project=credentials.project_id)
kohoshas_SMAP = client.query(query).result().to_dataframe()

In [13]:
query = """
SELECT
  shoki.id as shokikosho_id,
  shoki.kohosha_id,
  consts.name as ap_source,
  syi1.sei_plus as ap_kakutoku,
  syi2.sei_plus as mendan_tanto,
  shoki.kosho_setteibi,
  shoki.kosho_jisshibi,
  shoki.kosho_seq,
  shoki.saikosho_kaisu
FROM `temp-380708.live_rhs.shokikoshos` shoki
left join `temp-380708.live_company.syain` syi1 on shoki.ap_kakutoku = syi1.user_id
left join `temp-380708.live_company.syain` syi2 on shoki.mendan_tanto = syi2.user_id
left join (SELECT 
              code,
              name
             FROM `temp-380708.live_rhs.sys_consts`
             where group_code = 19) consts on shoki.ap_source = consts.code
where consts.name = "SMAP"
order by shoki.id
"""

client = bigquery.Client(credentials=credentials, project=credentials.project_id)
shokikosho_SMAP = client.query(query).result().to_dataframe()

In [15]:
query = """
SELECT distinct
  format_date('%Y/%m/%d',khs.created) as created,
  khs.id as kohosha_id,
  khs.seimei,
  consts.name as ap_source
FROM `temp-380708.live_rhs.kohoshas` khs
left join (SELECT 
              code,
              name
             FROM `temp-380708.live_rhs.sys_consts`
             where group_code = 19) consts on khs.ap_source = consts.code
full join (SELECT
             shoki.id as shokikosho_id,
             shoki.kohosha_id,
             consts.name as ap_source,
             syi1.sei_plus as ap_kakutoku,
             syi2.sei_plus as mendan_tanto,
             shoki.kosho_setteibi,
             shoki.kosho_jisshibi,
             shoki.kosho_seq,
             shoki.saikosho_kaisu
            FROM `temp-380708.live_rhs.shokikoshos` shoki
            left join `temp-380708.live_company.syain` syi1 on shoki.ap_kakutoku = syi1.user_id
            left join `temp-380708.live_company.syain` syi2 on shoki.mendan_tanto = syi2.user_id
            left join (SELECT 
                        code,
                        name
                       FROM `temp-380708.live_rhs.sys_consts`
                       where group_code = 19) consts on shoki.ap_source = consts.code
            where consts.name = "SMAP"
            order by shoki.id) shokikosho_SMAP on khs.id = shokikosho_SMAP.kohosha_id
where consts.name = "SMAP"
order by khs.id
"""

client = bigquery.Client(credentials=credentials, project=credentials.project_id)
SMAP_kohosha = client.query(query).result().to_dataframe()

 

In [22]:
query = """
SELECT 
  hon.id as honkosho_id,
  vhon.kohosha_id as kohosha_id,
  consts.name as ap_source,
  kgy.name as Client_name,
  syi1.sei_plus as kohosha_tanto,
  syi2.sei_plus as kigyo_tanto,
  ymi.name as yomi,
  format_date('%Y/%m/%d',hon.kosho_setteibi) as setteibi,
  format_date('%Y/%m/%d',hon.kosho_yoteibi) as yoteibi,
  format_date('%Y/%m/%d',hon.kosho_jisshibi) as jisshibi,
  format_date('%Y/%m/%d',hon.seiyakubi) as seiyakubi,
  hon.kosho_seq,
  vhon.saikosho_flag,
  case when an.kumite = 10 then '片手'
  　else '両手' end as kumite
FROM `temp-380708.live_rhs.honkoshos` hon
left join `temp-for-sandbox.ronzanmi__mart.vw_honkosho_saikoshos` vhon on hon.id = vhon.Honkosho_id
left join `temp-380708.live_rhs.ankens` an on vhon.anken_id = an.id
left join `temp-380708.live_rhs.kohoshas` khs on vhon.kohosha_id = khs.id
left join `temp-380708.live_rhs.kigyos` kgy on an.kigyo_id = kgy.id
left join `temp-380708.live_company.syain` syi1 on hon.kohosha_tanto = syi1.user_id
left join `temp-380708.live_company.syain` syi2 on an.kigyo_tanto = syi2.user_id
left join (SELECT 
              code,
              name
             FROM `temp-380708.live_rhs.sys_consts`
             where group_code = 19) consts on khs.ap_source = consts.code
left join (SELECT
              group_code,
              code,
              name
           FROM `temp-380708.live_rhs.sys_consts`
           where group_code = 53) ymi on hon.yomi = ymi.code
where hon.kosho_setteibi >= date '2019-09-30'
order by khs.id
"""

client = bigquery.Client(credentials=credentials, project=credentials.project_id)
SMAP_honkosho = client.query(query).result().to_dataframe()

In [23]:
SMAP_honkosho

,honkosho_id,kohosha_id,ap_source,Client_name,kohosha_tanto,kigyo_tanto,yomi,setteibi,yoteibi,jisshibi,seiyakubi,kosho_seq,saikosho_flag,kumite
0,17175,<NA>,None,None,前山諒,None,None,2023/06/13,2023/06/19,None,None,1,0,両手
1,9756,37,パートナー紹介,アグラジャパン（株）,大矢裕,大矢裕,None,2020/10/18,2020/11/04,None,None,1,0,両手
2,6832,59,パートナー紹介,（株）明和ｅテック,角田隆,大仲研,R：ロンザン成約済み・打診不可,2019/11/08,2019/11/15,2019/11/15,None,1,0,両手
3,11336,244,パートナー紹介,（株）鳥取メカシステム,大矢裕,大矢裕,None,2021/03/26,2021/04/06,2021/04/06,None,1,0,両手
4,8382,350,人事部経由,ワボウ電子（株）,大仲研,金子祐,None,2020/05/07,2020/05/11,2020/05/11,None,1,0,両手
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10689,17105,44512,その他,アポロトレイディング（株）,長谷川だ,長崎文,None,2023/06/06,2023/06/14,None,None,1,0,両手
10690,17144,44550,パートナー紹介,アルビス（株）,喜久里哲,大仲研,None,2023/06/09,2023/06/13,None,None,1,0,両手
10691,17142,44592,その他,アルビス（株）,古賀達,森永祐,None,2023/06/09,2023/06/09,None,None,1,0,片手
10692,17167,44634,顧問名鑑登録 解放者,（株）ショーシン,吉池直,石橋燎,None,2023/06/12,2023/06/14,None,None,1,0,片手


In [16]:
SMAP_kohosha

,created,kohosha_id,seimei,ap_source
0,2016/06/17,448,宮内 雄太郎,SMAP
1,2016/06/17,449,管野 正喜,SMAP
2,2016/06/17,450,降旗 宏之,SMAP
3,2016/06/17,451,大矢根 光次,SMAP
4,2016/06/17,452,中岡 卓,SMAP
...,...,...,...,...
6052,2023/06/13,44673,金崎 泰三,SMAP
6053,2023/06/13,44674,鈴木 達夫,SMAP
6054,2023/06/13,44675,半田 光弘,SMAP
6055,2023/06/13,44676,戸苅 義之,SMAP


In [12]:
kohoshas_SMAP

,created,kohosha_id,seimei,ap_source
0,2016/06/17,471,児玉 幹,SMAP
1,2016/06/17,487,園田 眞介,SMAP
2,2016/06/17,448,宮内 雄太郎,SMAP
3,2016/06/17,475,清水 俊一,SMAP
4,2016/06/17,459,二階 尚基,SMAP
...,...,...,...,...
6052,2023/06/13,44671,森 卓司,SMAP
6053,2023/06/13,44644,山田 晃代,SMAP
6054,2023/06/13,44664,廣木 肇,SMAP
6055,2023/06/13,44674,鈴木 達夫,SMAP


In [14]:
shokikosho_SMAP

,shokikosho_id,kohosha_id,ap_source,ap_kakutoku,mendan_tanto,kosho_setteibi,kosho_jisshibi,kosho_seq,saikosho_kaisu
0,12,968,SMAP,None,角田隆,2016-10-11,2016-10-19,1,0
1,22,979,SMAP,None,服部圭,2016-10-18,2016-10-18,1,0
2,59,1017,SMAP,None,長崎文,2016-11-01,2016-11-01,1,0
3,62,1023,SMAP,None,大塚洋,2016-10-28,2016-11-02,1,0
4,65,1026,SMAP,None,阿曽祐,2016-10-28,2016-10-31,1,0
...,...,...,...,...,...,...,...,...,...
7680,51702,44673,SMAP,浅野桃,浅野桃,2023-06-13,NaT,1,0
7681,51703,44674,SMAP,大塚洋,坂巻汐,2023-06-13,NaT,1,0
7682,51704,44675,SMAP,浅野桃,吉池直,2023-03-15,2023-03-15,1,0
7683,51705,44676,SMAP,浅野桃,吉武南,2023-06-13,NaT,1,0


In [5]:
#2020年1月以降の手上げ情報取得
SCOPES = ['https://www.googleapis.com/auth/spreadsheets.readonly',
          'https://www.googleapis.com/auth/spreadsheets']
json_path = r"C:\Users\suehara\Desktop\お転機BOX\ぱいそん練習\python_ss\credentials.json"
service = ps.get_auth(SCOPES,json_path)
SPREADSHEET_ID = '15qRR-yfJxgCh_TpXwBjBHNAnc8yAeTUCF0-Y_N6GoIo'
Sheet_NAME = '候補者状況!A'
Sheet_row = ":O"
RANGE_NAME = Sheet_NAME+Sheet_row
teageinfo = ps.get_ss(SPREADSHEET_ID,RANGE_NAME,service)
teageinfo = teageinfo[["ID","登録日時","処理フラグ","フリ先","手あげ日付","担当","手あげ"]]

In [6]:
teageinfo["ID"] = teageinfo["ID"].fillna(0)

In [7]:
#2019年9月以降の手上げ情報取得
SPREADSHEET_ID = '1fOGhqvCoER3YYv2npDUT1KAB6Fw9fYfQ29Vf52p5Nts'
Sheet_NAME = '～19.12.31!A'
Sheet_row = ":O"
RANGE_NAME = Sheet_NAME+Sheet_row
pastteageinfo = ps.get_ss(SPREADSHEET_ID,RANGE_NAME,service)
pastteageinfo = pastteageinfo[["ID","登録日時","処理フラグ","フリ先","手あげ日付","担当","手あげ"]]

In [8]:
pastteageinfo["ID"] = pastteageinfo["ID"].fillna(0)
teageinfo = pd.concat([pastteageinfo,teageinfo],axis=0,ignore_index=True)
teageinfoA = teageinfo.copy()

In [9]:
#2020年1月以降の手上げ情報取得
SCOPES = ['https://www.googleapis.com/auth/spreadsheets.readonly',
          'https://www.googleapis.com/auth/spreadsheets']
json_path = r"C:\Users\suehara\Desktop\お転機BOX\ぱいそん練習\python_ss\credentials.json"
service = ps.get_auth(SCOPES,json_path)
SPREADSHEET_ID = '15qRR-yfJxgCh_TpXwBjBHNAnc8yAeTUCF0-Y_N6GoIo'
Sheet_NAME = 'マスタ!A'
Sheet_row = ":B"
RANGE_NAME = Sheet_NAME+Sheet_row
tktoroku = ps.get_ss(SPREADSHEET_ID,RANGE_NAME,service)

In [10]:
tktoroku["ID"] = tktoroku["ID"].astype(int)
tktoroku = tktoroku.rename(columns={"ID":"tenki_id"})
tktoroku = tktoroku.query('tenki_id >= 46273')

In [11]:
teageinfoA = teageinfoA.rename(columns={"ID":"tenki_id"})
tktoroku["tenki_id"] = tktoroku["tenki_id"].astype(str)
tktoroku = pd.merge(tktoroku,teageinfoA,how="left",on=("tenki_id"))

In [12]:
tktoroku = tktoroku.rename(columns={"tenki_id":"転機ID","フリ先":"事業部","処理フラグ":"結果"})
tktoroku = tktoroku[["転機ID","created_at","結果","事業部","手あげ日付","担当","手あげ"]]

In [13]:
tktoroku.dtypes

転機ID          object
created_at    object
結果            object
事業部           object
手あげ日付         object
担当            object
手あげ           object
dtype: object

In [14]:

tktoroku

,転機ID,created_at,結果,事業部,手あげ日付,担当,手あげ
0,46273,2019/10/01,共有,ロンザン,2019/11/06,増田 智洋,増田 智洋
1,46274,2019/10/01,None,None,None,None,None
2,46275,2019/10/01,共有,ロンザン,2020/05/21,江頭 悠大,江頭 悠大
3,46276,2019/10/01,None,None,None,None,None
4,46277,2019/10/01,None,None,None,None,None
...,...,...,...,...,...,...,...
50685,105716,2023/06/12,None,None,None,None,None
50686,105717,2023/06/12,None,None,None,None,None
50687,105718,2023/06/12,None,None,None,None,None
50688,105719,2023/06/12,None,None,None,None,None


In [15]:

tktoroku.replace([np.inf, -np.inf], np.nan, inplace=True)
tktoroku.fillna('', inplace=True)

tktoroku = tktoroku.values.tolist()

In [16]:
tktoroku

[['46273', '2019/10/01', '共有', 'ロンザン', '2019/11/06', '増田 智洋', '増田 智洋'],
 ['46274', '2019/10/01', '', '', '', '', ''],
 ['46275', '2019/10/01', '共有', 'ロンザン', '2020/05/21', '江頭 悠大', '江頭 悠大'],
 ['46276', '2019/10/01', '', '', '', '', ''],
 ['46277', '2019/10/01', '', '', '', '', ''],
 ['46278', '2019/10/01', '', '', '', '', ''],
 ['46279', '2019/10/01', '', '', '', '', ''],
 ['46280', '2019/10/01', '', '', '', '', ''],
 ['46281', '2019/10/01', '共有', 'ロンザン', '2019/10/01', '笹原 啓佑', '2課'],
 ['46282', '2019/10/01', '', '', '', '', ''],
 ['46283', '2019/10/01', '共有', 'ロンザン', '2019/10/01', '大谷 昌宜', '3課'],
 ['46284', '2019/10/01', '', '', '', '', ''],
 ['46285', '2019/10/01', '共有', 'ロンザン', '2019/10/01', '大嶋 皓章', '3課'],
 ['46286', '2019/10/01', '共有', 'ロンザン', '2019/10/01', '高見澤 裕太', '4課'],
 ['46287', '2019/10/01', '', '', '', '', ''],
 ['46288', '2019/10/01', '', '', '', '', ''],
 ['46289', '2019/10/01', 'NG', '送信NG', '', '', ''],
 ['46290', '2019/10/01', '共有', 'ロンザン', '2019/10/07', '花田 和明', '花田 和

In [17]:
#スプレッドシートに記載する
SPREADSHEET_ID = '1ofer5LQADQ9ppKrtQWWbOyhyEmcrBx_c_O4UmVFe3wU'
Sheet_NAME = '登録!A'
Sheet_row = "2"
RANGE_NAME = Sheet_NAME+Sheet_row
ps.update_ss(SPREADSHEET_ID,RANGE_NAME,tktoroku,service)

## 初期交渉

In [18]:
rzshokikosho_query = """
SELECT
  shk.kohosha_id,
  shk.tenki_id,
  consts.name as ap_source,
  khs.birth_year,
  shk.annual_income,
  syi2.sei_plus as ap_kakutokusha,
  case when syi1.sei_plus is not null then syi1.sei_plus
  else shk.mendan_tanto end as mendan_tanto,
  format_date('%Y/%m/%d',shk.kosho_setteibi) as kosho_setteibi,
  format_date('%Y/%m/%d',shk.kosho_yoteibi) as kosho_yoteibi,
  format_date('%Y/%m/%d',shk.kosho_jisshibi) as kosho_jisshibi,
  shk.saikosho_kaisu
  FROM `temp-380708.live_rhs.shokikoshos` shk
  left join `temp-380708.live_rhs.kohoshas` khs on shk.kohosha_id = khs.id
  left join `temp-380708.live_company.syain` syi1 on shk.mendan_tanto = syi1.user_id
  left join `temp-380708.live_company.syain` syi2 on shk.ap_kakutoku = syi2.user_id
  left join (SELECT 
              code,
              name
             FROM `temp-380708.live_rhs.sys_consts`
             where group_code = 19) consts on shk.ap_source = consts.code
  where shk.kosho_setteibi >= date '2019-09-30'
  and consts.name = '転機社長名鑑'
  and shk.saikosho_kaisu = 0
  order by shk.kosho_setteibi
"""

client = bigquery.Client(credentials=credentials, project=credentials.project_id)
rzshokikosho = client.query(rzshokikosho_query).result().to_dataframe()


In [19]:
teageinfo = teageinfo.rename(columns={"ID":"tenki_id"})


In [20]:
rzshokikosho["tenki_id"] = rzshokikosho["tenki_id"].astype(str)
rzshokikosho = pd.merge(rzshokikosho,teageinfo,on=("tenki_id"),how=("left"))

In [21]:
rzshokikosho = rzshokikosho[["kohosha_id","tenki_id","ap_source","birth_year","annual_income",
                             "ap_kakutokusha","mendan_tanto","kosho_setteibi","kosho_yoteibi",
                             "kosho_jisshibi","saikosho_kaisu","手あげ"]]

In [22]:
rzshokikosho["tenki_id"] = rzshokikosho["tenki_id"].astype(str)
rzshokikosho["kohosha_id"] = rzshokikosho["kohosha_id"].astype(str)
rzshokikosho["birth_year"] = rzshokikosho["birth_year"].astype(str)
rzshokikosho["annual_income"] = rzshokikosho["annual_income"].astype(str)
rzshokikosho["birth_year"] = rzshokikosho["birth_year"].str.replace('<NA>', '')
rzshokikosho["tenki_id"] = rzshokikosho["tenki_id"].str.replace('<NA>', '')

rzshokikosho["kosho_setteibi"] = rzshokikosho["kosho_setteibi"].astype(str) 
rzshokikosho["kosho_yoteibi"] = rzshokikosho["kosho_yoteibi"].astype(str)
rzshokikosho["kosho_jisshibi"] = rzshokikosho["kosho_jisshibi"].astype(str)
rzshokikosho["saikosho_kaisu"] = rzshokikosho["saikosho_kaisu"].astype(str)
rzshokikosho["kosho_jisshibi"] = rzshokikosho["kosho_jisshibi"].str.replace('None', '')

In [23]:
rzshokikosho.replace([np.inf, -np.inf], np.nan, inplace=True)
rzshokikosho.fillna('', inplace=True)

rzshokikosho = rzshokikosho.values.tolist()

In [24]:
#スプレッドシートに記載する
SCOPES = ['https://www.googleapis.com/auth/spreadsheets.readonly',
          'https://www.googleapis.com/auth/spreadsheets']
json_path = r"C:\Users\suehara\Desktop\お転機BOX\ぱいそん練習\python_ss\credentials.json"
service = ps.get_auth(SCOPES,json_path)
SPREADSHEET_ID = '1ofer5LQADQ9ppKrtQWWbOyhyEmcrBx_c_O4UmVFe3wU'
Sheet_NAME = '初期交渉!A'
Sheet_row = "2"
RANGE_NAME = Sheet_NAME+Sheet_row
ps.update_ss(SPREADSHEET_ID,RANGE_NAME,rzshokikosho,service)

## 本交渉データ

In [25]:
rzhonkosho_query = """
SELECT 
  hks.id as honkosho_id,
  hks.anken_id as anken_id,
  an.kohosha_id as kohosha_id,
  khs.tenki_id as tenki_id,
  consts.name as ap_source,
  khs.birth_year,
  khs.annual_income,
  format_date('%Y/%m/%d',hks.kosho_setteibi) as setteibi,
  format_date('%Y/%m/%d',hks.kosho_yoteibi) as yoteibi,
  format_date('%Y/%m/%d',hks.kosho_jisshibi) as jisshibi,
  syi1.sei_plus as kohosha_tanto,
  syi2.sei_plus as kigyo_tanto,
  hks.kosho_seq
FROM `temp-380708.live_rhs.honkoshos` hks
left join `temp-for-sandbox.ronzanmi__mart.vw_honkosho_saikoshos` vhon on hks.id = vhon.Honkosho_id
left join `temp-380708.live_rhs.ankens` an on vhon.anken_id = an.id
left join `temp-380708.live_rhs.kohoshas` khs on vhon.kohosha_id = khs.id
left join `temp-380708.live_company.syain` syi1 on hks.kohosha_tanto = syi1.user_id
left join `temp-380708.live_company.syain` syi2 on an.kigyo_tanto = syi2.user_id
left join (SELECT 
              code,
              name
             FROM `temp-380708.live_rhs.sys_consts`
             where group_code = 19) consts on khs.ap_source = consts.code
where hks.kosho_setteibi >= date '2019-09-30'
and consts.name = '転機社長名鑑'
and hks.kosho_seq = 1
order by hks.kosho_setteibi
"""

client = bigquery.Client(credentials=credentials, project=credentials.project_id)
rzhonkosho = client.query(rzhonkosho_query).result().to_dataframe()

In [26]:
rzhonkosho

,honkosho_id,anken_id,kohosha_id,tenki_id,ap_source,birth_year,annual_income,setteibi,yoteibi,jisshibi,kohosha_tanto,kigyo_tanto,kosho_seq
0,6471,6031,3175,<NA>,転機社長名鑑,1959,900.0,2019/10/01,2019/10/08,2019/10/08,宇野憲,安田信,1
1,6481,6040,13545,<NA>,転機社長名鑑,1961,620.0,2019/10/02,2019/10/11,None,花田和,吉山史,1
2,6480,6039,13477,<NA>,転機社長名鑑,1963,1300.0,2019/10/02,2019/10/23,None,花田和,堀汐,1
3,6475,6034,13316,<NA>,転機社長名鑑,1962,1850.0,2019/10/02,2019/10/08,2019/10/08,大谷昌,長松大,1
4,6483,6042,13339,<NA>,転機社長名鑑,1962,1100.0,2019/10/02,2019/10/08,2019/10/08,小野佑,長松大,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4694,17153,16320,44138,104986,転機社長名鑑,1982,800.0,2023/06/09,2023/06/21,None,加藤諒,長松大,1
4695,17148,16315,34213,89454,転機社長名鑑,1969,900.0,2023/06/09,2023/06/13,None,喜久里哲,長谷川大,1
4696,17133,16300,21117,67379,転機社長名鑑,1964,1200.0,2023/06/09,2023/06/16,None,松下将,松下将,1
4697,17159,16324,29575,80921,転機社長名鑑,1965,800.0,2023/06/09,2023/06/25,None,菊地晶,柘植一,1


In [27]:
rzhonkosho["honkosho_id"] = rzhonkosho["honkosho_id"].astype(str) 
rzhonkosho["anken_id"] = rzhonkosho["anken_id"].astype(str) 
rzhonkosho["kohosha_id"] = rzhonkosho["kohosha_id"].astype(str) 
rzhonkosho["tenki_id"] = rzhonkosho["tenki_id"].astype(str) 
rzhonkosho["birth_year"] = rzhonkosho["birth_year"].astype(str) 
rzhonkosho["tenki_id"] = rzhonkosho["tenki_id"].str.replace('<NA>', '')
rzhonkosho["birth_year"] = rzhonkosho["birth_year"].str.replace('<NA>', '')

rzhonkosho["setteibi"] = rzhonkosho["setteibi"].astype(str) 
rzhonkosho["yoteibi"] = rzhonkosho["yoteibi"].astype(str)
rzhonkosho["jisshibi"] = rzhonkosho["jisshibi"].astype(str)
rzhonkosho["jisshibi"] = rzhonkosho["jisshibi"].str.replace('None', '')

In [28]:

rzhonkosho.replace([np.inf, -np.inf], np.nan, inplace=True)
rzhonkosho.fillna('', inplace=True)

rzhonkosho = rzhonkosho.values.tolist()

In [29]:
#スプレッドシートに記載する
SPREADSHEET_ID = '1ofer5LQADQ9ppKrtQWWbOyhyEmcrBx_c_O4UmVFe3wU'
Sheet_NAME = '設定!A'
Sheet_row = "2"
RANGE_NAME = Sheet_NAME+Sheet_row
ps.update_ss(SPREADSHEET_ID,RANGE_NAME,rzhonkosho,service)

## レイノスKPI

In [30]:
scteage = teageinfo.query('フリ先 == "レイノス"')
scteage = scteage[["tenki_id","登録日時","フリ先","手あげ日付","担当"]]

In [31]:
scteage

,tenki_id,登録日時,フリ先,手あげ日付,担当
44,167,2016/04/16,レイノス,2016/04/16,
47,172,2016/04/25,レイノス,2016/04/25,
50,175,2016/04/26,レイノス,2016/04/26,
51,176,2016/04/27,レイノス,2016/04/27,
60,187,2016/04/28,レイノス,2016/04/28,
...,...,...,...,...,...
96546,105481,2023/06/06,レイノス,2023/06/08,長坂 真太郎
96547,105482,2023/06/06,レイノス,2023/06/09,小幡 美登
96551,105486,2023/06/06,レイノス,2023/06/07,上大園 剛
96597,105533,2023/06/08,レイノス,2023/06/09,上大園 剛


In [32]:
schonkosho_query = """
SELECT
  rc.tenki_id,
	can.ap_kakutoku as shokikosho_setteibi,
	rc.first_nmen_date as first_jisshi_date,
	can.nmen_zisshi as shokikosho_jisshibi,
	CASE
		WHEN com.settei8 is not null THEN com.settei8 
		WHEN com.settei7 is not null THEN com.settei7
		WHEN com.settei6 is not null THEN com.settei6 
		WHEN com.settei5 is not null THEN com.settei5 
		WHEN com.settei4 is not null THEN com.settei4 
		WHEN com.settei3 is not null THEN com.settei3 
		WHEN com.settei2 is not null THEN com.settei2
		WHEN com.settei1 is not null THEN com.settei1 ELSE null
	END as honkosho_setteibi,
	CASE
		WHEN com.zisshi8 is not null THEN com.zisshi8
		WHEN com.zisshi7 is not null THEN com.zisshi7 
		WHEN com.zisshi6 is not null THEN com.zisshi6 
		WHEN com.zisshi5 is not null THEN com.zisshi5 
		WHEN com.zisshi4 is not null THEN com.zisshi4 
		WHEN com.zisshi3 is not null THEN com.zisshi3 
		WHEN com.zisshi2 is not null THEN com.zisshi2
		WHEN com.zisshi1 is not null THEN com.zisshi1 ELSE null
	END as honkosho_jissibi,
	com.seiyaku,
	can.kouhosya_tantou as kohosha_tanto,
	can.client_tantou as kigyo_tanto,
	can.mendan_tantou as mendan_tanto,
	CASE
		WHEN can.zisshi LIKE 'キャンセル%' THEN 'CXL' 
		WHEN can.zisshi LIKE '%実施%' THEN '実施' ELSE '-' 
	END AS CXL
FROM `temp-380708.live_sugarcrm52.rc_race_candidate_management` rc
left join `temp-380708.live_saltcrm.candidate` can on can.sugarid = rc.candidate_no
left join `temp-380708.live_saltcrm.company` com on com.candidate_id = can.id
where rc.tenki_id is not null 
AND rc.deleted = 0
and can.ap_kakutoku >= date '2019-10-01'
order by can.ap_kakutoku
"""

client = bigquery.Client(credentials=credentials, project=credentials.project_id)
schonkosho = client.query(schonkosho_query).result().to_dataframe()

In [33]:
schonkosho["tenki_id"] = schonkosho["tenki_id"].astype(str)

In [34]:
schonkosho = pd.merge(scteage,schonkosho,how="right",on=("tenki_id"))

In [35]:
schonkosho = schonkosho.query('フリ先 == "レイノス"')

In [36]:
schonkosho

,tenki_id,登録日時,フリ先,手あげ日付,担当,shokikosho_setteibi,first_jisshi_date,shokikosho_jisshibi,honkosho_setteibi,honkosho_jissibi,seiyaku,kohosha_tanto,kigyo_tanto,mendan_tanto,CXL
2,46327,2019/10/01,レイノス,2019/10/02,菊池 武二郎,2019-10-01,2019-10-07,2019-10-07,NaT,NaT,NaT,宇野 憲人,None,宇野 憲人,実施
12,46165,2019/09/28,レイノス,2019/09/28,小高 拓,2019-10-01,2019-10-11,NaT,NaT,NaT,NaT,小高拓,None,小高拓,-
13,46214,2019/09/29,レイノス,2019/09/29,小高 拓,2019-10-01,2019-10-08,2019-10-08,NaT,NaT,NaT,小高拓,None,小高拓,実施
14,45935,2019/09/23,レイノス,2019/09/23,リーダークラス＆課責リーダー,2019-10-01,2019-10-10,2019-10-10,NaT,NaT,NaT,樋口貴治,None,樋口貴治,実施
15,46159,2019/09/28,レイノス,2019/09/28,小高 拓,2019-10-01,2019-10-16,2019-10-16,NaT,NaT,NaT,小高拓,None,小高拓,実施
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9473,96460,2022/06/15,レイノス,2022/06/17,白間 友康,2023-06-09,2022-06-23,NaT,NaT,NaT,NaT,白間友康,None,中澤聖,-
9474,42555,2019/07/08,レイノス,2019/07/08,小高 拓,2023-06-09,2019-07-24,NaT,NaT,NaT,NaT,小高拓,None,木村祐人,-
9475,91623,2022/02/11,レイノス,2022/02/15,福塚 将司,2023-06-09,2022-05-25,NaT,NaT,NaT,NaT,福塚将司,None,今井皓太,-
9476,970,2016/09/04,レイノス,2016/09/04,,2023-06-09,2016-09-16,NaT,NaT,NaT,NaT,森翔平,None,藤田優希,-


In [37]:
schonkosho["first_jisshi_date"] = schonkosho["first_jisshi_date"].astype(str)
schonkosho["shokikosho_setteibi"] = schonkosho["shokikosho_setteibi"].astype(str)
schonkosho["shokikosho_jisshibi"] = schonkosho["shokikosho_jisshibi"].astype(str)
schonkosho["honkosho_setteibi"] = schonkosho["honkosho_setteibi"].astype(str)
schonkosho["honkosho_jissibi"] = schonkosho["honkosho_jissibi"].astype(str)
schonkosho["seiyaku"] = schonkosho["seiyaku"].astype(str)

schonkosho["shokikosho_setteibi"] = schonkosho["shokikosho_setteibi"].str.replace('NaT', '')
schonkosho["shokikosho_jisshibi"] = schonkosho["shokikosho_jisshibi"].str.replace('NaT', '')
schonkosho["honkosho_setteibi"] = schonkosho["honkosho_setteibi"].str.replace('NaT', '')
schonkosho["honkosho_jissibi"] = schonkosho["honkosho_jissibi"].str.replace('NaT', '')
schonkosho["seiyaku"] = schonkosho["seiyaku"].str.replace('NaT', '')

In [38]:
schonkosho.replace([np.inf, -np.inf], np.nan, inplace=True)
schonkosho.fillna('', inplace=True)

schonkosho = schonkosho.values.tolist()

In [39]:
#スプレッドシートに記載する
SPREADSHEET_ID = '1ofer5LQADQ9ppKrtQWWbOyhyEmcrBx_c_O4UmVFe3wU'
Sheet_NAME = 'SCKPI!A'
Sheet_row = "2"
RANGE_NAME = Sheet_NAME+Sheet_row
ps.update_ss(SPREADSHEET_ID,RANGE_NAME,schonkosho,service)